# Diễn giải cụm khách hàng và kế hoạch marketing

Notebook này chuyển kết quả K-Means từ các label `Cluster 0`, `Cluster 1`, `Cluster 2` thành các nhóm khách hàng có ý nghĩa kinh doanh.

Mục tiêu:

- Đọc bảng khách hàng đã phân cụm và cluster profile.
- Đặt tên cho từng cụm dựa trên Recency, Frequency và Monetary.
- Tạo action plan marketing cho từng nhóm.
- Lưu bảng `data/processed/cluster_action_plan.csv` để dùng trong dashboard và form dự đoán khách hàng.

## 1. Load dữ liệu phân cụm

Bước đầu tiên là đọc kết quả từ model K-Means đã train ở phase trước. Bảng segmented chứa từng khách hàng và cluster của họ, còn bảng profile chứa thống kê tổng hợp theo cụm.

Đoạn code dưới đây import thư viện, đọc `data/processed/rfm_uk_segmented.csv` và `data/processed/cluster_profile.csv` từ thư mục `models/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SEGMENTED_PATH = PROCESSED_DIR / "rfm_uk_segmented.csv"
PROFILE_PATH = PROCESSED_DIR / "cluster_profile.csv"
ACTION_PLAN_PATH = PROCESSED_DIR / "cluster_action_plan.csv"

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12

rfm_segmented = pd.read_csv(SEGMENTED_PATH)
cluster_profile = pd.read_csv(PROFILE_PATH)

cluster_profile.round(2)

**Nhận xét:** ba cụm có đặc điểm khác nhau rõ ràng. Cluster 2 có Recency thấp nhất, Frequency và Monetary cao nhất nên phù hợp với nhóm VIP. Cluster 1 có Recency cao nhất và giá trị thấp nhất nên phù hợp với nhóm inactive hoặc at-risk. Cluster 0 nằm giữa hai nhóm còn lại, phù hợp với nhóm khách hàng tiềm năng/normal.

## 2. Tạo bảng diễn giải cụm

K-Means chỉ tạo ra label dạng số, vì vậy cần gán tên nhóm và ý nghĩa nghiệp vụ cho từng cluster. Việc đặt tên dựa trên profile RFM gốc, không dựa trực tiếp vào centroid đã scale.

Đoạn code dưới đây tạo bảng action plan cho từng cluster, gồm tên nhóm, mô tả hành vi, insight và chiến lược marketing đề xuất.

In [ ]:
cluster_actions = pd.DataFrame(
    [
        {
            "Cluster": 0,
            "SegmentName": "Khách hàng tiềm năng / thông thường",
            "ShortName": "Potential",
            "BusinessMeaning": (
                "Nhóm khách mua tương đối gần đây, có tần suất và mức chi tiêu ở mức trung bình. "
                "Họ chưa đạt mức VIP nhưng vẫn có khả năng phát triển nếu được chăm sóc đúng cách."
            ),
            "KeyInsight": (
                "Cluster 0 chiếm 39.17% khách hàng và tạo 25.22% doanh thu. "
                "Recency trung bình là 43.17 ngày, Frequency trung bình 3.43 hóa đơn, "
                "Monetary trung bình 1,190.74 GBP."
            ),
            "MarketingAction": (
                "Tập trung tăng tần suất mua bằng voucher cho lần mua tiếp theo, combo sản phẩm, "
                "gợi ý sản phẩm liên quan và chương trình tích điểm cơ bản."
            ),
        },
        {
            "Cluster": 1,
            "SegmentName": "Khách hàng ít hoạt động / có nguy cơ rời bỏ",
            "ShortName": "At-Risk",
            "BusinessMeaning": (
                "Nhóm khách đã lâu chưa quay lại, mua ít và đóng góp doanh thu thấp. "
                "Đây là nhóm cần chiến dịch tái kích hoạt nhưng không nên tiêu tốn quá nhiều chi phí chăm sóc."
            ),
            "KeyInsight": (
                "Cluster 1 chiếm 43.69% khách hàng nhưng chỉ tạo 8.19% doanh thu. "
                "Recency trung bình là 166.88 ngày, Frequency trung bình 1.36 hóa đơn, "
                "Monetary trung bình 346.77 GBP."
            ),
            "MarketingAction": (
                "Dùng chiến dịch win-back chi phí thấp như email nhắc quay lại, mã giảm giá giới hạn thời gian, "
                "khảo sát lý do không mua lại và ưu đãi tái kích hoạt."
            ),
        },
        {
            "Cluster": 2,
            "SegmentName": "Khách hàng VIP / trung thành",
            "ShortName": "VIP",
            "BusinessMeaning": (
                "Nhóm khách mua gần đây, quay lại thường xuyên và đóng góp phần lớn doanh thu. "
                "Đây là nhóm có giá trị cao nhất và cần được ưu tiên giữ chân."
            ),
            "KeyInsight": (
                "Cluster 2 chỉ chiếm 17.13% khách hàng nhưng tạo 66.59% doanh thu. "
                "Recency trung bình là 17.87 ngày, Frequency trung bình 13.41 hóa đơn, "
                "Monetary trung bình 7,187.67 GBP."
            ),
            "MarketingAction": (
                "Ưu tiên giữ chân bằng loyalty program, ưu đãi độc quyền, quyền mua sớm sản phẩm mới, "
                "chăm sóc cá nhân hóa và đề xuất sản phẩm dựa trên lịch sử mua."
            ),
        },
    ]
)

cluster_actions

## 3. Gộp profile và action plan

Bảng cuối cùng kết hợp thống kê cụm với tên nhóm và chiến lược marketing. Đây là bảng dễ đưa vào README, báo cáo hoặc dashboard.

Đoạn code dưới đây merge `cluster_profile` với `cluster_actions` để tạo bảng diễn giải hoàn chỉnh cho từng cụm.

In [ ]:
cluster_interpretation = cluster_profile.merge(cluster_actions, on="Cluster", how="left")

ordered_columns = [
    "Cluster",
    "SegmentName",
    "ShortName",
    "Customers",
    "CustomerPct",
    "Revenue",
    "RevenuePct",
    "RecencyMean",
    "RecencyMedian",
    "FrequencyMean",
    "FrequencyMedian",
    "MonetaryMean",
    "MonetaryMedian",
    "BusinessMeaning",
    "KeyInsight",
    "MarketingAction",
]

cluster_interpretation = cluster_interpretation[ordered_columns]
cluster_interpretation.round(2)

**Nhận xét:** bảng này cho thấy ba nhóm có vai trò khác nhau rõ ràng. Nhóm VIP nhỏ nhất về số lượng nhưng tạo phần lớn doanh thu; nhóm at-risk lớn nhất về số khách nhưng đóng góp doanh thu thấp; nhóm potential nằm giữa và là nhóm có thể nuôi dưỡng để tăng giá trị.

## 4. Biểu đồ profile cụm

Các biểu đồ dưới đây giúp nhìn nhanh sự khác biệt giữa các cụm theo số khách hàng, doanh thu và ba chỉ số RFM.

Đoạn code dưới đây vẽ số khách hàng theo từng nhóm khách hàng.

In [ ]:
plot_profile = cluster_interpretation.sort_values("Cluster").copy()

ax = sns.barplot(data=plot_profile, x="ShortName", y="Customers", hue="ShortName", legend=False)
ax.set_title("Customer Count by Segment")
ax.set_xlabel("Segment")
ax.set_ylabel("Customers")
ax.bar_label(ax.containers[0], fmt="%.0f", padding=3)
plt.tight_layout()
plt.show()

**Nhận xét:** nhóm At-Risk có 1,711 khách hàng, chiếm 43.69% tổng khách. Nhóm Potential có 1,534 khách hàng, chiếm 39.17%. Nhóm VIP nhỏ hơn, có 671 khách hàng, tương đương 17.13%.

Đoạn code dưới đây vẽ tỷ trọng doanh thu theo từng nhóm khách hàng.

In [ ]:
ax = sns.barplot(data=plot_profile, x="ShortName", y="RevenuePct", hue="ShortName", legend=False)
ax.set_title("Revenue Share by Segment")
ax.set_xlabel("Segment")
ax.set_ylabel("Revenue Share (%)")
ax.bar_label(ax.containers[0], fmt="%.2f%%", padding=3)
plt.tight_layout()
plt.show()

**Nhận xét:** nhóm VIP chỉ chiếm 17.13% khách hàng nhưng tạo 66.59% doanh thu. Ngược lại, nhóm At-Risk chiếm 43.69% khách hàng nhưng chỉ tạo 8.19% doanh thu.

Đoạn code dưới đây vẽ ba biểu đồ so sánh giá trị trung bình của Recency, Frequency và Monetary giữa các nhóm.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ("RecencyMean", "Average Recency", "Days"),
    ("FrequencyMean", "Average Frequency", "Invoices"),
    ("MonetaryMean", "Average Monetary", "GBP"),
]

for ax, (col, title, ylabel) in zip(axes, metrics):
    sns.barplot(data=plot_profile, x="ShortName", y=col, hue="ShortName", legend=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Segment")
    ax.set_ylabel(ylabel)
    ax.bar_label(ax.containers[0], fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()

**Nhận xét:** nhóm VIP có Recency trung bình thấp nhất, 17.87 ngày, Frequency trung bình cao nhất, 13.41 hóa đơn, và Monetary trung bình cao nhất, 7,187.67 GBP. Nhóm At-Risk ngược lại có Recency trung bình 166.88 ngày, Frequency 1.36 hóa đơn và Monetary 346.77 GBP.

Đoạn code dưới đây vẽ scatter plot Recency và Monetary theo nhóm. Biểu đồ này hữu ích cho dashboard vì nó cho thấy nhóm khách nào mua gần đây và đóng góp doanh thu cao.

In [ ]:
segment_lookup = cluster_actions.set_index("Cluster")["ShortName"].to_dict()
rfm_segmented["Segment"] = rfm_segmented["Cluster"].map(segment_lookup)

ax = sns.scatterplot(
    data=rfm_segmented,
    x="Recency",
    y="Monetary",
    hue="Segment",
    alpha=0.65,
)
ax.set_title("Recency vs Monetary by Segment")
ax.set_xlabel("Recency")
ax.set_ylabel("Monetary (GBP)")
plt.tight_layout()
plt.show()

## 5. Lưu action plan

Sau khi kiểm tra profile và biểu đồ, bảng action plan được lưu lại để dashboard có thể hiển thị tên nhóm, insight và chiến lược chăm sóc tương ứng với cluster model dự đoán.

Đoạn code dưới đây lưu bảng diễn giải cụm thành `data/processed/cluster_action_plan.csv`.

In [ ]:
cluster_interpretation.to_csv(ACTION_PLAN_PATH, index=False)

print(f"Saved {ACTION_PLAN_PATH}")